In [1]:
import sys
sys.path.append("src/")

from pathlib import Path
import tqdm 
import random
import time
from concurrent.futures import ThreadPoolExecutor
import threading
import ipaddress

from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np

from datagathering.proxies import decodo, dummy
from proxyrecord import IpRecord
from utils import filecache

In [2]:
OUTPUT_FILE = Path("data/prod-withifconfig-decodo.json")


In [3]:
num_steps = 500
base_timeout = 1  # Minimum timeout in seconds
jitter_range = 2  # Jitter will be between 0 and 0.5 seconds
lock = threading.Lock()
max_workers = 4

In [4]:
data_from_disk = filecache.read_list_from_file(OUTPUT_FILE, IpRecord)
results = dict()
doubles = 0

for i in data_from_disk:
    results[i.ip] = i
    
print(len(results))

538


In [5]:
def process_one(_, pbar):
    global doubles
    
    iprecord = decodo.get_proxy_address()
    if not iprecord:
        pbar.update(1)
        return False

    # Check for duplicates safely
    with lock:
        if iprecord.ip in results:
            doubles += 1
            pbar.set_postfix(doubles=doubles)
            pbar.update(1)
            time.sleep(base_timeout + random.uniform(0, jitter_range))
            return False

    iprecord.load_whois()

    # Update shared structures + file, again under lock
    with lock:
        results[iprecord.ip] = iprecord
        filecache.write_list_to_file(OUTPUT_FILE, list(results.values()))

    time.sleep(base_timeout + random.uniform(0, jitter_range))
    pbar.update(1)
    return True


In [6]:
if True:
    doubles = 0
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        with tqdm.tqdm(total=num_steps) as pbar:
            # pass pbar into the worker so it can update the bar and postfix
            list(executor.map(lambda i: process_one(i, pbar), range(num_steps)))

 41%|████      | 204/500 [06:13<06:58,  1.41s/it, doubles=185]

Trying to load whois for ipv6 address


 56%|█████▌    | 279/500 [08:20<08:17,  2.25s/it, doubles=251]

Trying to load whois for ipv6 address


100%|██████████| 500/500 [14:58<00:00,  1.80s/it, doubles=459]


In [7]:
print(len(results))

579


In [8]:
for i in tqdm.tqdm(results):
    item = results[i]
    had_abuse = "amount" in item.abuse
    item.load_abuse()
    
    if not had_abuse:
        time.sleep(base_timeout + random.uniform(0, jitter_range))
filecache.write_list_to_file(OUTPUT_FILE, list(results.values()))

In [9]:
for i in tqdm.tqdm(results):
    item = results[i]
    if item.rdns == "<none>":
        item.rdns = ""
    had_rdns = item.rdns != ""
    item.load_rdns()
    
    if not had_rdns:
        time.sleep(0.05)
filecache.write_list_to_file(OUTPUT_FILE, list(results.values()))

100%|██████████| 579/579 [00:47<00:00, 12.23it/s]
